In [ ]:
import torch
import corner
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
import json

from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 50
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=MEDIUM_SIZE)  # fontsize of the figure title

In [ ]:
def import_statistics(stats_path: str):
    """
    Extracting the mean and standard deviation for all the parameters in the `stats_path` file.
    Args:
        stats_path (str): Path to the file where the statistics are saved.
    Returns:
        (torch.tensor, torch.tensor): Mean and standard deviation for the parameters in the `stats_path` file.
    """
    std_list = []
    mean_list = []
    max_list = []
    min_list = []
    with open(stats_path, "r") as json_file:
        data = json.load(json_file)
    for key, value in data.items():
        std_list.append(value["std"])
        mean_list.append(value["mean"])
        max_list.append(value["max"])
        min_list.append(value["min"])
    mean = np.array(mean_list)
    std = np.array(std_list)
    max_list = np.array(max_list)
    min_list = np.array(min_list)
    return mean, std, max_list, min_list

In [ ]:
#stats_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/statistics_train_10K_7param.json"
stats_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/statistics_train_1K_7param.json"
#stats_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/statistics_train_10K_5param.json"
#stats_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/statistics_train_1K_5param.json"

#directory_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/exp_2_ensemble5comp_10rounds_firstround10k_retrain_from_scratch_5param"
#directory_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/exp_4_ensemble5comp_10rounds_allrounds1k_retrain_from_scratch_5param"
#directory_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/exp_6_ensemble5comp_10rounds_firstround10k_retrain_from_scratch_7param"
#directory_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/exp_7_ensemble5comp_10rounds_allrounds1k_retrain_from_scratch_7param_test_sample"
#directory_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/exp_8_ensemble5comp_10rounds_allrounds1k_retrain_from_scratch_7param"
#directory_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/exp_10_ensemble5comp_10rounds_firstround10k_retrain_from_scratch_7param_updated_atnf"
directory_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/exp_11_ensemble5comp_10rounds_allrounds1k_retrain_from_scratch_7param_updated_atnf"
#directory_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/exp_12_ensemble5comp_10rounds_allrounds1k_retrain_from_scratch_7param_test_sample_updated_atnf"
#directory_path = "/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/exp_13_ensemble5comp_10rounds_allrounds1k_retrain_from_scratch_7param_updated_atnf_no_ppdot_fluxes"

#If the tsnpe was perform to estimate a test sample then you can overplot the ground truth by setting test_sample = True and set the ground truth to true_value parameter.
test_sample = False
true_values = [13.25,0.75,-0.6,0.3,-2,26.9,0.5]

observed_posterior_round = []
coverage_round = []

n_rounds = 10
colors = ['tab:cyan', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', 'tab:blue', 'tab:olive', 'tab:gray']
colors_2 = ["#fde725", "dodgerblue", "#440154"]
for i in range(n_rounds):
    observed_posterior_round.append(torch.load(f"{directory_path}/round_{i}/samples_posterior_{i}.pt").detach().cpu().numpy())
    coverage_round.append(np.load(f"{directory_path}/round_{i}/coverage_probability.npy"))

In [ ]:
exp_number = [
    1,
    2,
    3,
    4,
    5,
    6,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    19,
    20,
    21,
]

In [ ]:
posteriors_atnf = []

for n in exp_number:
    
    posteriors_atnf.append(
        torch.load(f"/home/celsa/Documents/MAGNESIA_population_synthesis/data/paper_results/graber_etal_2024/samples/samples_exp_{n}_atnf.pt")
        .detach()
        .cpu()
        .numpy()
        .T
    )
posteriors_ensemble_atnf = np.concatenate(posteriors_atnf, axis=1)

In [ ]:
mean, std, par_max, par_min = import_statistics(stats_path)

In [ ]:
if np.shape(observed_posterior_round)[2] == 7:
    limits = [[12.0, 14.0], [0.1, 1.0], [-1.5, -0.3], [0.1, 1.0], [-3, -0.5],[24.6,28.6],[0.1,1]]
    x_ticks = [[12.0,13.0, 14.0], [0.1,0.5, 0.9], [-1.5,-1, -0.5], [0.1, 0.5,0.9], [-3, -2,-1],[24.6,26.6,28.5],[0.2,0.5,1]]
    parameter_labels = [
        r"$\mu_{\log B}$",
        r"$\sigma_{\log B}$",
        r"$\mu_{\log P}$",
        r"$\sigma_{\log P}$",
        r"$a_{\rm late}$",
        r"$\mu_{\log L_0}$",
        r"$\alpha$"
    
    ]
elif np.shape(observed_posterior_round)[2] == 5:
    limits = [[12.0, 14.0], [0.1, 1.0], [-1.5, -0.3], [0.1, 1.0], [-3, -0.5]]
    x_ticks = [[12.0,13.0, 14.0], [0.1,0.5, 0.9], [-1.5,-1, -0.5], [0.1, 0.5,0.9], [-3, -2,-1]]
    parameter_labels = [
        r"$\mu_{\log B}$",
        r"$\sigma_{\log B}$",
        r"$\mu_{\log P}$",
        r"$\sigma_{\log P}$",
        r"$a_{\rm late}$"
    
    ]
n_param = np.shape(observed_posterior_round)[2]

In [ ]:
credibility_level = np.linspace(0, 1, 12)

In [ ]:
# Define the number of rounds and parameters
n_rounds = len(observed_posterior_round)
n_param = len(parameter_labels)

# Set fixed y-axis limits (adjust based on your data)
y_fixed_limits = [0, 17]  # Adjust this limit based on your data

# Create larger subplots and add space for round numbers
fig, axs = plt.subplots(n_rounds, n_param+1, figsize=(27, 15),gridspec_kw={'hspace': 0, 'wspace': 0.2})

# Loop through the parameters and rounds
for j in range(n_param):
    for i in range(n_rounds):
        observed_posterior = observed_posterior_round[i]
        observed_posterior = observed_posterior * std[0:n_param] + mean[0:n_param]

        # Set the color for each round
        curve_color = 'tab:blue'  # You can change this color if needed
        
        # Select the correct axis for each round and parameter
        ax = axs[i][j]
        # Calculate the 95% confidence interval using percentiles
        lower_bound = np.percentile(observed_posterior.T[j], 2.5)
        upper_bound = np.percentile(observed_posterior.T[j], 97.5)

        # Add grey shading for the 95% confidence interval
        ax.axvspan(lower_bound, upper_bound, color='tab:gray', alpha=0.3)

        # If the number of parameter is equal to 5 plot also the results from Graber et al 2024.
        if n_param ==5:
            ax.hist(
            posteriors_ensemble_atnf[j],
            bins=32,
            color='black',
            histtype="step",
            linewidth=2,
            density=True,
        )
        
        ax.hist(
            observed_posterior.T[j],
            bins=32,
            color=curve_color,
            edgecolor=curve_color,
            linewidth=2,
            histtype="step",
            density=True,
        )
        
        for past_round in range(0,i):
            observed_posterior_past_round = observed_posterior_round[past_round]
            observed_posterior_past_round = observed_posterior_past_round * std[0:n_param] + mean[0:n_param]

            ax.hist(
                observed_posterior_past_round.T[j],
                bins=32,
                color='grey',
                edgecolor='grey',
                alpha = 0.5,
                linewidth=1.5,
                histtype="step",
                density=True,
            )
        if test_sample:
             # Add vertical line for true values
            ax.axvline(
                true_values[j],
                color='tab:orange',  # Color of the vertical line
                linestyle='--',  # Dashed line style
                linewidth=2, 
            )
        
        # Only add x-axis tick marks (without labels) for all rows except the last
        if i < n_rounds - 1 and i>0:
            ax.tick_params(axis='x', which='major', length=10, width=1.5,top=True, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  # For major ticks
            ax.tick_params(axis='x', which='minor', length=6, width=1,top=True, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  
            ax.set_xticks(x_ticks[j])
            ax.minorticks_on()
        elif i == 0:
            ax.tick_params(axis='x', which='major', length=10, width=1.5,top=False, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  # For major ticks
            ax.tick_params(axis='x', which='minor', length=6, width=1,top=False, labeltop=False, bottom=True, labelbottom=False, direction = 'inout',grid_color='r', grid_alpha=0.5)  
            ax.set_xticks(x_ticks[j])
            ax.minorticks_on()
        else:
            ax.tick_params(axis='x', which='major', length=10, width=1.5,top=True, labeltop=False, bottom=True, labelbottom=True, direction = 'inout')  # For major ticks
            ax.tick_params(axis='x', which='minor', length=6, width=1,top=True, labeltop=False, bottom=True, labelbottom=True, direction = 'inout')  
            ax.set_xlabel(parameter_labels[j], fontsize=MEDIUM_SIZE)
            ax.set_xticks(x_ticks[j])
            ax.minorticks_on()
            
            
        # Remove y-axis ticks for all subplots
        ax.set_yticks([])
        #ax.tick_params(axis='x', labelsize=SMALL_SIZE-4)

        # Only set titles in the top row
        if i == 0:
            ax.set_title(parameter_labels[j], fontsize=MEDIUM_SIZE)

        # Add the round number next to each row, rotated parallel to the y-axis
        fig.text(0.12, 0.83 - (i / 1.28 - 0.18) / n_rounds, f'Round {i+1}', va='center', ha='center',
                 fontsize=18, rotation='vertical')

        # Set limits for the x-axis
        ax.set_xlim(limits[j])
        # Automagically incresing the ylimit axis to a 30% more than the data described.
        ax.margins(y=0.3)
           
# Plot the coverage probability in the last column
for i in range(n_rounds):
    ax = axs[i][n_param]  # Last column for coverage plot
    for past_round in range(0,i):
        ax.plot(
            credibility_level,
            coverage_round[past_round],
            linestyle="-",
            color='tab:grey',
            linewidth=1,
            alpha=0.3,
            rasterized=True
        )
    ax.plot(
            credibility_level,
            coverage_round[i],
            linestyle="-",
            color='tab:blue',
            linewidth=3,
            rasterized=True
        )
    ax.plot(
        credibility_level,
        credibility_level,
        linestyle="-",
        color="black",
        linewidth=2,
        alpha=1,
        rasterized=True,
        label=r"Well-calibrated",
    )
    #ax.grid()
    # Set limits for the coverage probability plot
    ax.set_xlim(0, 1.0)
    ax.set_ylim(0, 1.0)
    coverage_ticks = [0.1,0.5,0.9]
    if i < n_rounds - 1 and i>0:
        ax.grid(which='both')
        ax.tick_params(axis='x', which='major', length=10, width=1.5,top=True, labeltop=False, bottom=True, labelbottom=False, direction = 'inout')  # For major ticks
        ax.set_xticks(coverage_ticks)
        

    elif i == 0:
        ax.grid(which='both')
        ax.tick_params(axis='x', which='major', length=10, width=1.5, top=False, labeltop=False, bottom=True, labelbottom=False)
        ax.set_xticks(coverage_ticks)

    else:
        ax.grid(which='both')
        ax.set_xlabel(r"Credibility level $1 - \alpha$", fontsize=SMALL_SIZE)
        ax.set_xticks(coverage_ticks)

    if i == 0:
        ax.set_title(r"Credibility level $1 - \alpha$", fontsize=SMALL_SIZE)

    credibility_ticks = [0.1,0.5,0.9]
    ax.set_yticks(credibility_ticks,credibility_ticks, fontsize = SMALL_SIZE-4)
    ax.tick_params(axis='y', which='major', length=10, width=1.5,left=False, labelleft=False, right=False, labelright=False, direction = 'out')  # For major ticks
    
    if i == n_rounds-1:
        ax.tick_params(axis='y', which='major', length=10, width=1.5,left=False, labelleft=False, right=True, labelright=True, direction = 'out')  # For major ticks
        ax.set_yticks(credibility_ticks,credibility_ticks, fontsize = SMALL_SIZE)
fig.text(0.91, 0.5, 'Coverage Probability', va='center', ha='center', fontsize=SMALL_SIZE, rotation=270)


# Adjust layout: no vertical space but with horizontal space
#plt.subplots_adjust(hspace=0, wspace=0.2)
plt.savefig('/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/plots/pardo_et_al_2024/exp_13.pdf',bbox_inches="tight")
# Save and show the figure
plt.show()

### Corner plot

In [ ]:
SMALL_SIZE = 14
MEDIUM_SIZE = 20
BIGGER_SIZE = 30

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
#We take the posterior distribution from round `round_corner`.
round_corner = 5
observed_samples = observed_posterior_round[round_corner]

In [ ]:
observed_samples = observed_samples * std[0:n_param] + mean[0:n_param]

# Saving the best estimated parameters and the 95% CI into the log.txt file.
quantile = np.quantile(observed_samples, [0.025, 0.5, 0.975], axis=0)

range_param = [[par_min[v], par_max[v]] for v in range(n_param)]

param_median = quantile[1, :]

figure = corner.corner(
    observed_samples,
    bins=32,
    labels=parameter_labels,
    #range=range_param,
    color='k',
    quantiles=[0.025, 0.5, 0.975],
    levels=(
        1 - np.exp(-0.5),
        1 - np.exp(-2),
        1 - np.exp(-9.0 / 2.0),
    ),  # 1, 2 and 3 sigma levels
    show_titles=True,
    title_kwargs={"fontsize": 18},
)

corner.overplot_lines(figure, param_median, color="dodgerblue")

corner.overplot_points(
    figure,
    param_median[None],
    marker="o",
    color="dodgerblue",
)
plt.savefig('/home/celsa/Documents/Jupyter-notebooks/data/pardo_et_al_2023/plots/pardo_et_al_2024/exp_11_corner_plot.pdf',dpi=400)


plt.show()

## Corelation between the parameters  

In [ ]:
from scipy.stats import pearsonr

# Null hypothesis is that there is no correlation. Therefore, a p-value close to 0 indicated strong evidence of correlation.
L_0 = observed_samples[:,5]
parameter = observed_samples[:,6]
corr_coefficient, p_value = pearsonr(L_0, parameter)
print("Correlation Coefficient:", corr_coefficient, p_value)

### Parameter estimates: 95 and 50 CI

In [ ]:
# Compute the quantiles (2.5th, 50th, and 97.5th percentiles)
quantile = np.quantile(observed_samples, [0.025, 0.5, 0.975], axis=0)

# Compute the median as the 50th percentile
medians = quantile[1, :]

# Compute the deviations of the 2.5th and 97.5th percentiles from the mean (median)
deviation_lower = medians - quantile[0, :]
deviation_upper = quantile[2, :] - medians

# Display the results as median + deviation from the mean
for i, label in enumerate(parameter_labels):
    print(f"{label}: {medians[i]:.2f} -{deviation_lower[i]:.2f} +{deviation_upper[i]:.2f}")


## Comparison between this work and Graber et al 

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(25, 8))


axs = axs.ravel()

for j in range(5):
    axs[j].hist(
        observed_samples.T[j],
        bins=32,
        color='tab:blue',
        linewidth=4,
        histtype="step",
        density=True,
        rasterized = True,
        label = 'This work'
    ) 
    axs[j].hist(
        posteriors_ensemble_atnf[j],
        bins=32,
        color='k',
        linewidth=4,
        histtype="step",
        density=True,
        rasterized = True,
        label = 'Graber et al'

    )

    axs[j].set_xlabel(parameter_labels[j], fontsize=MEDIUM_SIZE)
    axs[j].set_xlim(limits[j])
    axs[j].set_yticks([])
   

plt.tight_layout()

plt.show()